In [19]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy.stats import entropy

In [20]:
# 1. Load Data
iris = load_iris()
X = iris.data
y = iris.target

In [21]:
# Petal Length (index 2) and Petal Width (index 3)
petal_lengths = X[:, 2]
petal_widths = X[:, 3]

In [22]:
# --- Core Math Functions (Same as before) ---
def calc_entropy(labels):
    if len(labels) == 0: return 0
    counts = np.bincount(labels)
    probs = counts[counts > 0] / len(labels)
    return entropy(probs, base=2)

def calc_information_gain(feature_values, labels, threshold):
    parent_entropy = calc_entropy(labels)
    left_mask = feature_values <= threshold
    right_mask = feature_values > threshold
    
    left_labels = labels[left_mask]
    right_labels = labels[right_mask]
    
    weight_left = len(left_labels) / len(labels)
    weight_right = len(right_labels) / len(labels)
    
    child_entropy = (weight_left * calc_entropy(left_labels)) + (weight_right * calc_entropy(right_labels))
    return parent_entropy - child_entropy

In [23]:
# --- Step 1: The First Split (Isolating Setosa) ---
split_1_feature = petal_lengths
split_1_threshold = 2.3 # Established from the previous run

In [24]:
# --- Step 2: Filtering Data for the Second Split ---
# We only care about the data that went to the RIGHT node (Versicolor & Virginica)
right_node_mask = split_1_feature > split_1_threshold
X_subset = X[right_node_mask]
y_subset = y[right_node_mask]

# Focus on Petal Width for the next split
subset_petal_widths = X_subset[:, 3]

In [25]:
# --- Step 3: Finding the Best Second Split ---
thresholds_2 = np.unique(subset_petal_widths)
info_gains_2 = [calc_information_gain(subset_petal_widths, y_subset, t) for t in thresholds_2]

best_idx_2 = np.argmax(info_gains_2)
split_2_threshold = thresholds_2[best_idx_2]
max_gain_2 = info_gains_2[best_idx_2]

In [26]:
# --- Plotting the 2D Decision Boundaries ---
plt.figure(figsize=(10, 7))

# Scatter the original data
scatter = plt.scatter(petal_lengths, petal_widths, c=y, cmap='viridis', edgecolor='k', s=60, alpha=0.8)

# Draw First Split (Vertical Line on Petal Length)
plt.axvline(x=split_1_threshold, color='red', linestyle='-', linewidth=2, 
            label=f'Split 1: Length <= {split_1_threshold}cm')

# Draw Second Split (Horizontal Line on Petal Width, starting AFTER the first split)
# We use xmin to ensure the line only draws in the "Right Node" area
plt.hlines(y=split_2_threshold, xmin=split_1_threshold, xmax=max(petal_lengths)+0.5, 
           colors='blue', linestyles='--', linewidth=2, 
           label=f'Split 2: Width <= {split_2_threshold}cm')

In [29]:
# Formatting
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Decision Tree Boundaries: 2D Feature Space')

# Custom Legend
handles, labels = scatter.legend_elements(prop="colors", alpha=0.6)
class_labels = [iris.target_names[0], iris.target_names[1], iris.target_names[2]]
legend1 = plt.legend(handles, class_labels, title="Classes", loc="lower right")
plt.gca().add_artist(legend1)
plt.legend(loc="upper left")

plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("--- Right Node (Versicolor & Virginica) Analysis ---")
print(f"Node Entropy before Split 2: {calc_entropy(y_subset):.3f} bits")
print(f"Best Split Threshold (Petal Width): <= {split_2_threshold} cm")
print(f"Information Gain from Split 2: {max_gain_2:.3f} bits")

/var/folders/ll/97k3bpnx19sgpbqws5x_x77c0000gn/T/ipykernel_28236/1858330796.py:11: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(loc="upper left")


--- Right Node (Versicolor & Virginica) Analysis ---
Node Entropy before Split 2: 1.000 bits
Best Split Threshold (Petal Width): <= 1.7 cm
Information Gain from Split 2: 0.690 bits
